# Пайплайн транскрибации 1% аудио с `openai/whisper-large-v3` (Hugging Face)

Этот ноутбук читает `data/calls-3-quarantine.jsonl`, берёт воспроизводимую выборку 1% записей, загружает `.wav` по полю `path` и транскрибирует аудио в текст с помощью модели `openai/whisper-large-v3`.

> Важно: модель очень тяжёлая. На CPU будет медленно и может не хватить RAM. Для практического запуска рекомендуется GPU (CUDA).

## 1) Установка зависимостей

Установите библиотеки (можно выполнить прямо в ноутбуке). Если у вас GPU NVIDIA, поставьте CUDA-сборку PyTorch по инструкции с https://pytorch.org/get-started/locally/.

Пакеты:
- `transformers`, `accelerate`
- `torch`, `torchaudio`
- `pandas`, `pyarrow`, `tqdm`

Важно: чтобы `pipeline(...)` мог читать аудио по пути к файлу, в системе должен быть установлен `ffmpeg` (это не Python-пакет).


In [1]:
# ЯЧЕЙКА "Установка зависимостей" (переписать код ячейки целиком)

# %pip install -U transformers accelerate pandas pyarrow tqdm
# torch/torchaudio ставьте CUDA-сборкой отдельно (см. pytorch.org)

import json
import math
import os
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import torch
import transformers
from tqdm.auto import tqdm
from transformers import AutoProcessor, pipeline

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))

torch: 2.10.0+cu128
transformers: 4.57.3
cuda available: True
cuda device: NVIDIA GeForce RTX 4060


## 2) Конфигурация путей и параметров

Здесь задаются:
- путь к JSONL с метаданными,
- доля выборки (1%),
- seed для воспроизводимости,
- параметры chunking/batch для Whisper,
- опциональный remap путей (если `Z:\...` недоступен в вашей среде).

In [ ]:
# ЯЧЕЙКА "Конфигурация путей и параметров" (заменить блок выбора device/dtype)

# ---- Пути ----
PROJECT_DIR = Path.cwd()
DATA_JSONL = PROJECT_DIR / "data" / "calls-3-quarantine.jsonl"

# ---- Выборка ----
sample_frac = 0.002
seed = 42

# ---- ASR параметры ----
# model_id = "openai/whisper-large-v3-turbo"
model_id = "openai/whisper-large-v3"
chunk_length_s = 30
stride_length_s = 5
batch_size = 6
max_new_tokens = 384

force_language: Optional[str] = "russian"

return_timestamps = True

path_prefix_from: Optional[str] = None
path_prefix_to: Optional[str] = None

# ---- Выход ----
OUT_DIR = PROJECT_DIR / "data"
OUT_DIR.mkdir(parents=True, exist_ok=True)
output_base = OUT_DIR / "transcripts_whisper_large_v3_1pct"

# ---- CUDA ONLY ----
cuda_device_index = 0
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA недоступна. Поставьте CUDA-сборку torch/torchaudio и выберите правильный kernel."
    )

device = cuda_device_index  # ВАЖНО: только CUDA
torch_dtype = torch.float16  # ВАЖНО: fp16 на GPU

# Опционально (ускорение на Ampere+)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("DATA_JSONL:", DATA_JSONL)
print("exists:", DATA_JSONL.exists())
print("device:", device)
print("dtype:", torch_dtype)


@dataclass
class RunConfig:
    data_jsonl: str
    sample_frac: float
    seed: int
    model_id: str
    chunk_length_s: int
    stride_length_s: int
    batch_size: int
    max_new_tokens: int
    force_language: Optional[str]
    return_timestamps: bool
    path_prefix_from: Optional[str]
    path_prefix_to: Optional[str]
    device: str
    torch_dtype: str


run_config = RunConfig(
    data_jsonl=str(DATA_JSONL),
    sample_frac=sample_frac,
    seed=seed,
    stride_length_s=stride_length_s,
    model_id=model_id,
    chunk_length_s=chunk_length_s,
    batch_size=batch_size,
    max_new_tokens=max_new_tokens,
    force_language=force_language,
    return_timestamps=return_timestamps,
    path_prefix_from=path_prefix_from,
    path_prefix_to=path_prefix_to,
    device=str(device),
    torch_dtype=str(torch_dtype),
)

run_config

DATA_JSONL: d:\project\multicriteria-dialog-audit-ml\data\calls-3-quarantine.jsonl
exists: True
device: 0
dtype: torch.float16


RunConfig(data_jsonl='d:\\project\\multicriteria-dialog-audit-ml\\data\\calls-3-quarantine.jsonl', sample_frac=0.002, seed=42, model_id='openai/whisper-large-v3', chunk_length_s=30, stride_length_s=5, batch_size=6, max_new_tokens=384, force_language='russian', return_timestamps=True, path_prefix_from=None, path_prefix_to=None, device='0', torch_dtype='torch.float16')

In [3]:
import os
import shutil
import subprocess

FFMPEG_BIN_DIR = os.environ.get("FFMPEG_BIN_DIR")

if FFMPEG_BIN_DIR:
    os.environ["PATH"] = FFMPEG_BIN_DIR + os.pathsep + os.environ.get("PATH", "")

ffmpeg_path = shutil.which("ffmpeg")
print("ffmpeg:", ffmpeg_path)

if ffmpeg_path:
    try:
        r = subprocess.run(
            ["ffmpeg", "-version"], capture_output=True, text=True, check=False
        )
        first_line = (r.stdout.splitlines()[:1] or r.stderr.splitlines()[:1] or [""])[0]
        if first_line:
            print(first_line)
    except Exception as e:
        print("ffmpeg -version failed:", e)
else:
    print(
        "ffmpeg не найден в PATH. Если вы только что установили ffmpeg — перезапустите kernel/VS Code. "
        "Либо задайте FFMPEG_BIN_DIR (папка bin с ffmpeg.exe) через env var или прямо в этой ячейке."
    )

ffmpeg: Z:\ffmpeg\bin\ffmpeg.EXE
ffmpeg version N-123094-g561f37c023-20260301 Copyright (c) 2000-2026 the FFmpeg developers


## 3) Загрузка метаданных из JSONL и базовая валидация

Читаем `calls-3-quarantine.jsonl`, проверяем обязательные колонки и фильтруем записи без доступного аудиофайла (после remap, если включён).

In [4]:
def iter_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f"JSON decode error at line {line_no}: {e}")


def remap_path(p: str) -> str:
    if (
        path_prefix_from
        and path_prefix_to
        and p.lower().startswith(path_prefix_from.lower())
    ):
        # сохранить остаток пути как есть (Windows case-insensitive)
        return path_prefix_to + p[len(path_prefix_from) :]
    return p


rows = list(iter_jsonl(DATA_JSONL))
df = pd.DataFrame(rows)

required_cols = {"id", "path", "duration_sec"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Отсутствуют обязательные колонки в JSONL: {sorted(missing)}")

df["path_remap"] = df["path"].astype(str).map(remap_path)
df["path_exists"] = df["path_remap"].map(lambda x: Path(x).exists())

n_total = len(df)
n_exists = int(df["path_exists"].sum())

print(f"Всего записей: {n_total}")
print(f"Файлы найдены: {n_exists} ({n_exists / max(1, n_total):.1%})")

# Оставим только доступные файлы для транскрибации

df_ok = df[df["path_exists"]].copy()
df_bad = df[~df["path_exists"]].copy()

print("df_ok:", df_ok.shape)
print("df_bad:", df_bad.shape)

df_ok.head(3)

Всего записей: 51744
Файлы найдены: 51744 (100.0%)
df_ok: (51744, 7)
df_bad: (0, 7)


,id,path,file_name,duration_sec,size_bytes,path_remap,path_exists
0,677adc71c4b90b9ea4a88396658e2b180de27d03,Z:\calls\dataset\2026\01\14\1768365555.19884.wav,1768365555.19884.wav,19.42,310764,Z:\calls\dataset\2026\01\14\1768365555.19884.wav,True
1,c4b517189c0d8d765f2ab4061366be7a42c5572c,Z:\calls\dataset\2026\01\14\1768365653.19890.wav,1768365653.19890.wav,16.00,256044,Z:\calls\dataset\2026\01\14\1768365653.19890.wav,True
2,980d0e76dc2245b8de80c468ed5ef6da6b8a1a15,Z:\calls\dataset\2026\01\14\1768365826.19904.wav,1768365826.19904.wav,27.32,437164,Z:\calls\dataset\2026\01\14\1768365826.19904.wav,True


## 4) Выборка 1% записей (воспроизводимое семплирование)

Семплируем 1% от *доступных* файлов. Защита от нулевого размера: берём минимум 1 запись.

In [5]:
if len(df_ok) == 0:
    raise RuntimeError(
        "Не найдено ни одного доступного аудиофайла. "
        "Если пути начинаются с Z:\\..., задайте path_prefix_from/path_prefix_to в конфиге."
    )

n = max(1, int(math.ceil(len(df_ok) * sample_frac)))

df_sample = df_ok.sample(n=n, random_state=seed).copy()
# Для удобства — сначала короткие файлы
if "duration_sec" in df_sample.columns:
    df_sample = df_sample.sort_values("duration_sec", ascending=True)

print(f"Доступных файлов: {len(df_ok)}")
print(f"Семпл 1%: n={len(df_sample)}")

df_sample[["id", "path_remap", "duration_sec"]].head(10)

Доступных файлов: 51744
Семпл 1%: n=104


,id,path_remap,duration_sec
9284,1bde4b40f2b0ff25a14f54b1e5e0941ed4f63e57,Z:\calls\dataset\2026\01\13\1768312679.18042.wav,10.22
13857,0c5e746ac073d02c3c154f2fd1a74eedf84b40f2,Z:\calls\dataset\2025\12\29\1766991400.592561.wav,10.36
21163,dae13b07c26401809cd6c1ad491d8dbbe22d08a7,Z:\calls\dataset\2025\12\24\1766575090.439812.wav,10.38
23400,ba7763a5eb85bcc1d7c444dd2e54afdb77ecb51c,Z:\calls\dataset\2025\12\24\1766584502.461159.wav,10.94
42793,26b2b46c349e0aa803f7a1edfb3db421f01747cf,Z:\calls\dataset\2025\12\19\1766145984.238130.wav,11.02
147,2db34b8a2de8437964a88b2a89422775cc39f507,Z:\calls\dataset\2026\01\14\1768370558.21466.wav,11.06
36024,4e75fddb021f9fa0052409a4abcec6bad2735a44,Z:\calls\dataset\2025\12\22\1766406416.310675.wav,11.28
29313,e5ac96648251dd8a032ea5e3387ef4fc021b3118,Z:\calls\dataset\2025\12\23\1766493749.383321.wav,11.38
45416,9450df88675f07c12142bffeeed0a9e1bedce4e7,Z:\calls\dataset\2025\12\18\1766039757.126564.wav,12.08
1276,9f3dd761958c48c8cd6661415f6bb5c976bd6044,Z:\calls\dataset\2026\01\14\1768380147.32673.wav,12.26


## 5) Чтение аудио по пути (ffmpeg)

В этом ноутбуке аудио читается прямо внутри `transformers.pipeline("automatic-speech-recognition")` по пути к файлу.

Если видите ошибку вида `ffmpeg was not found...`, значит `ffmpeg` не установлен или не доступен в `PATH`.


## 6) Инициализация `openai/whisper-large-v3` через Hugging Face

Используем `transformers.pipeline("automatic-speech-recognition")`. Для Whisper можно передавать `generate_kwargs` (язык, task).

In [6]:
_generate_kwargs = {
    "task": "transcribe",
    "max_new_tokens": max_new_tokens,
    "num_beams": 1,
    "compression_ratio_threshold": 1.35,
    "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
    "condition_on_prev_tokens": False,
    "logprob_threshold": -1.0,
    "no_speech_threshold": 0.6,
}

if force_language:
    _generate_kwargs["language"] = force_language

pipe = pipeline(
    task="automatic-speech-recognition",
    model=model_id,
    torch_dtype=torch_dtype,
    device=device,
    chunk_length_s=chunk_length_s,
    stride_length_s=stride_length_s,
    batch_size=batch_size,
    return_timestamps=return_timestamps,
    generate_kwargs=_generate_kwargs,
)

pipe

'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /openai/whisper-large-v3/resolve/main/config.json (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1000)')))"), '(Request ID: 809d52ff-9157-4705-abe9-870621899c2c)')' thrown while requesting HEAD https://huggingface.co/openai/whisper-large-v3/resolve/main/config.json
Retrying in 1s [Retry 1/5].
`torch_dtype` is deprecated! Use `dtype` instead!


Device set to use cuda:0
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


## 7) Батчевый пайплайн транскрибации (chunking, fp16/bf16, GPU)

Проходим по 1% выборке, грузим аудио, прогоняем через pipeline и собираем результаты. Ошибки не роняют весь прогон — пишем их в поле `error`.

In [7]:
def normalize_text(text: Optional[str]) -> str:
    if not text:
        return ""
    # trim + схлопнуть пробелы
    return " ".join(str(text).strip().split())


results = []

for row in tqdm(df_sample.itertuples(index=False), total=len(df_sample)):
    audio_path = str(getattr(row, "path_remap"))
    call_id = str(getattr(row, "id"))
    duration_sec = float(getattr(row, "duration_sec", np.nan))

    t0 = time.perf_counter()

    if not Path(audio_path).exists():
        results.append(
            {
                "id": call_id,
                "path": str(getattr(row, "path")),
                "path_remap": audio_path,
                "duration_sec": duration_sec,
                "text": "",
                "language": None,
                "wall_time_sec": float(time.perf_counter() - t0),
                "rtf": None,
                "error": "file not found",
            }
        )
        continue

    try:
        out = pipe(audio_path)

        text = normalize_text(out.get("text") if isinstance(out, dict) else str(out))

        print(out)

        wall = float(time.perf_counter() - t0)
        rtf = (wall / duration_sec) if (duration_sec and duration_sec > 0) else None

        results.append(
            {
                "id": call_id,
                "path": str(getattr(row, "path")),
                "path_remap": audio_path,
                "duration_sec": duration_sec,
                "text": text,
                "out": out,
                "language": out.get("language") if isinstance(out, dict) else None,
                "wall_time_sec": wall,
                "rtf": rtf,
                "error": None,
            }
        )
    except Exception as e:
        wall = float(time.perf_counter() - t0)
        results.append(
            {
                "id": call_id,
                "path": str(getattr(row, "path")),
                "path_remap": audio_path,
                "duration_sec": duration_sec,
                "text": "",
                "out": None,
                "language": None,
                "wall_time_sec": wall,
                "rtf": None,
                "error": f"ASR failed: {e}",
            }
        )


df_out = pd.DataFrame(results)
df_out.head()

  0%|          | 0/104 [00:00<?, ?it/s]

{'text': ' Здравствуйте. Вызываемый вами абонент сейчас не отвечает. Возможно, на его телефоне включен беззвучный режим, или он не может взять трубку. Пожалуйста, попробуйте.', 'chunks': [{'timestamp': (0.0, 4.04), 'text': ' Здравствуйте. Вызываемый вами абонент сейчас не отвечает.'}, {'timestamp': (4.56, 7.22), 'text': ' Возможно, на его телефоне включен беззвучный режим,'}, {'timestamp': (7.22, 10.22), 'text': ' или он не может взять трубку. Пожалуйста, попробуйте.'}]}
{'text': ' Потому что там авангард, но есть оплата. Вызываемый недоступен. Вы звоните по срочному вопросу? Нет, но есть, которых не отчислили.', 'chunks': [{'timestamp': (0.0, 2.46), 'text': ' Потому что там авангард, но есть оплата.'}, {'timestamp': (2.84, 4.24), 'text': ' Вызываемый недоступен.'}, {'timestamp': (4.64, 6.38), 'text': ' Вы звоните по срочному вопросу?'}, {'timestamp': (6.38, 8.78), 'text': ' Нет, но есть, которых не отчислили.'}]}
{'text': ' Оставайтесь на линии.', 'chunks': [{'timestamp': (0.0, 2.0), 

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


{'text': ' Здравствуйте, вы позвонили в профессиональную коллекторскую организацию «Актив Бизнес Консалт». Если вы хотите связаться с нами, перезвоните по бесплатному.', 'chunks': [{'timestamp': (0.0, 8.26), 'text': ' Здравствуйте, вы позвонили в профессиональную коллекторскую организацию «Актив Бизнес Консалт».'}, {'timestamp': (8.94, 12.26), 'text': ' Если вы хотите связаться с нами, перезвоните по бесплатному.'}]}
{'text': ' Алло, да. Алло, здравствуйте. А я вот сегодня не на счет Яндекс.Карт. Да, минут через 10 буквально перезвоните. Да, хорошо.', 'chunks': [{'timestamp': (0.0, 0.92), 'text': ' Алло, да.'}, {'timestamp': (1.6, 2.36), 'text': ' Алло, здравствуйте.'}, {'timestamp': (2.62, 4.84), 'text': ' А я вот сегодня не на счет Яндекс.Карт.'}, {'timestamp': (7.26, 9.84), 'text': ' Да, минут через 10 буквально перезвоните.'}, {'timestamp': (10.24, 10.9), 'text': ' Да, хорошо.'}]}
{'text': ' Алло. Алло. Это было...', 'chunks': [{'timestamp': (0.0, 0.88), 'text': ' Алло.'}, {'timest

Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


...

Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


{'text': ' Абонент сейчас не может ответить на ваш звонок. Его телефон занят. Попробуйте перезвонить позднее. Если хотите отправить бесплатное смс-сообщение с просьбой перезвонить, нажмите 1. Спасибо. Всего доброго. Спасибо. Всего доброго. Спасибо.', 'chunks': [{'timestamp': (0.0, None), 'text': ' Абонент сейчас не может ответить на ваш звонок. Его телефон занят. Попробуйте перезвонить позднее. Если хотите отправить бесплатное смс-сообщение с просьбой перезвонить, нажмите 1. Спасибо. Всего доброго. Спасибо. Всего доброго. Спасибо.'}]}
{'text': ' Здравствуйте! Меня Антон, зовут, компания Виктория. Мы проводим форум в Екатеринбурге по стоматологии и ортодонтии. Скажите, пожалуйста, у вас маркетолог есть в компании? Нет. Нет, нет. А может быть, управляющему вашему было интересно? Да, не знаю. Можете отправить на dendogran. собачка янда круз ру информация указываю еще раз прошу прощения пропали для анны а отчество Там долго Нориковна. Все, записал. Спасибо вам большое. А отчество? Там долг

Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


{'text': ' Продолжаем дозваниваться. Оставайтесь на линии. ЗВОНОК ТЕЛЕФОНА Абонент не берет трубку. Попробуйте перезвонить позднее. Если вы хотите отправить ему бесплатное смс-сообщение с просьбой перезвонить, нажмите 1. Спасибо. Всего доброго.', 'chunks': [{'timestamp': (0.0, 2.8), 'text': ' Продолжаем дозваниваться. Оставайтесь на линии.'}, {'timestamp': (26.28, 2.8), 'text': ''}, {'timestamp': (28.2, 28.78), 'text': ' ЗВОНОК ТЕЛЕФОНА Абонент не берет трубку.'}, {'timestamp': (30.36, 31.0), 'text': ' Попробуйте перезвонить позднее.'}, {'timestamp': (36.94, 44.62), 'text': ' Если вы хотите отправить ему бесплатное смс-сообщение с просьбой перезвонить, нажмите 1.'}, {'timestamp': (47.0, None), 'text': ' Спасибо. Всего доброго.'}]}
{'text': ' Здравствуйте! На связи Зарина, бренд стильной женской и мужской одежды. Ваш звонок очень важен для нас. Оставайтесь на линии, вам ответит первый освободившийся оператор. Продолжая разговор, вы даете согласие на обработку персональных данных. В целя

,id,path,path_remap,duration_sec,text,out,language,wall_time_sec,rtf,error
0,1bde4b40f2b0ff25a14f54b1e5e0941ed4f63e57,Z:\calls\dataset\2026\01\13\1768312679.18042.wav,Z:\calls\dataset\2026\01\13\1768312679.18042.wav,10.22,Здравствуйте. Вызываемый вами абонент сейчас н...,{'text': ' Здравствуйте. Вызываемый вами абоне...,None,2.440084,0.238756,None
1,0c5e746ac073d02c3c154f2fd1a74eedf84b40f2,Z:\calls\dataset\2025\12\29\1766991400.592561.wav,Z:\calls\dataset\2025\12\29\1766991400.592561.wav,10.36,"Потому что там авангард, но есть оплата. Вызыв...","{'text': ' Потому что там авангард, но есть оп...",None,1.254509,0.121092,None
2,dae13b07c26401809cd6c1ad491d8dbbe22d08a7,Z:\calls\dataset\2025\12\24\1766575090.439812.wav,Z:\calls\dataset\2025\12\24\1766575090.439812.wav,10.38,Оставайтесь на линии.,"{'text': ' Оставайтесь на линии.', 'chunks': [...",None,0.539700,0.051994,None
3,ba7763a5eb85bcc1d7c444dd2e54afdb77ecb51c,Z:\calls\dataset\2025\12\24\1766584502.461159.wav,Z:\calls\dataset\2025\12\24\1766584502.461159.wav,10.94,"Алло. Алло, здравствуйте, Надежда. Меня зовут ...","{'text': ' Алло. Алло, здравствуйте, Надежда. ...",None,1.444742,0.132061,None
4,26b2b46c349e0aa803f7a1edfb3db421f01747cf,Z:\calls\dataset\2025\12\19\1766145984.238130.wav,Z:\calls\dataset\2025\12\19\1766145984.238130.wav,11.02,"Добрый день. Добрый день. Подскажите, с кем мо...",{'text': ' Добрый день. Добрый день. Подскажит...,None,1.146300,0.104020,None


In [8]:
df_out = pd.DataFrame(results)
df_out.head()

,id,path,path_remap,duration_sec,text,out,language,wall_time_sec,rtf,error
0,1bde4b40f2b0ff25a14f54b1e5e0941ed4f63e57,Z:\calls\dataset\2026\01\13\1768312679.18042.wav,Z:\calls\dataset\2026\01\13\1768312679.18042.wav,10.22,Здравствуйте. Вызываемый вами абонент сейчас н...,{'text': ' Здравствуйте. Вызываемый вами абоне...,None,2.440084,0.238756,None
1,0c5e746ac073d02c3c154f2fd1a74eedf84b40f2,Z:\calls\dataset\2025\12\29\1766991400.592561.wav,Z:\calls\dataset\2025\12\29\1766991400.592561.wav,10.36,"Потому что там авангард, но есть оплата. Вызыв...","{'text': ' Потому что там авангард, но есть оп...",None,1.254509,0.121092,None
2,dae13b07c26401809cd6c1ad491d8dbbe22d08a7,Z:\calls\dataset\2025\12\24\1766575090.439812.wav,Z:\calls\dataset\2025\12\24\1766575090.439812.wav,10.38,Оставайтесь на линии.,"{'text': ' Оставайтесь на линии.', 'chunks': [...",None,0.539700,0.051994,None
3,ba7763a5eb85bcc1d7c444dd2e54afdb77ecb51c,Z:\calls\dataset\2025\12\24\1766584502.461159.wav,Z:\calls\dataset\2025\12\24\1766584502.461159.wav,10.94,"Алло. Алло, здравствуйте, Надежда. Меня зовут ...","{'text': ' Алло. Алло, здравствуйте, Надежда. ...",None,1.444742,0.132061,None
4,26b2b46c349e0aa803f7a1edfb3db421f01747cf,Z:\calls\dataset\2025\12\19\1766145984.238130.wav,Z:\calls\dataset\2025\12\19\1766145984.238130.wav,11.02,"Добрый день. Добрый день. Подскажите, с кем мо...",{'text': ' Добрый день. Добрый день. Подскажит...,None,1.146300,0.104020,None


## 8) Постобработка текста и метрики производительности (RTF)

$RTF = \frac{t_{wall}}{t_{audio}}$ — чем меньше, тем быстрее (RTF<1 означает быстрее реального времени).

In [9]:
ok_mask = df_out["error"].isna()

print("Успешно:", int(ok_mask.sum()), "/", len(df_out))

if ok_mask.any():
    stats = df_out.loc[ok_mask, "rtf"].dropna().astype(float)
    if len(stats) > 0:
        print("RTF mean:", float(stats.mean()))
        print("RTF median:", float(stats.median()))

# покажем несколько примеров
show_cols = ["id", "duration_sec", "wall_time_sec", "rtf", "text", "error"]
df_out[show_cols].head(5)

Успешно: 104 / 104
RTF mean: 0.22074572849580573
RTF median: 0.1108325958772079


,id,duration_sec,wall_time_sec,rtf,text,error
0,1bde4b40f2b0ff25a14f54b1e5e0941ed4f63e57,10.22,2.440084,0.238756,Здравствуйте. Вызываемый вами абонент сейчас н...,None
1,0c5e746ac073d02c3c154f2fd1a74eedf84b40f2,10.36,1.254509,0.121092,"Потому что там авангард, но есть оплата. Вызыв...",None
2,dae13b07c26401809cd6c1ad491d8dbbe22d08a7,10.38,0.539700,0.051994,Оставайтесь на линии.,None
3,ba7763a5eb85bcc1d7c444dd2e54afdb77ecb51c,10.94,1.444742,0.132061,"Алло. Алло, здравствуйте, Надежда. Меня зовут ...",None
4,26b2b46c349e0aa803f7a1edfb3db421f01747cf,11.02,1.146300,0.104020,"Добрый день. Добрый день. Подскажите, с кем мо...",None


## 9) Сохранение результатов (JSONL/Parquet) + лог ошибок

Сохраняем:
- Parquet (быстро, удобно для аналитики)
- JSONL (строковый формат)
- отдельный файл с ошибками
- метаданные запуска (config + список id семпла)

In [10]:
out_parquet = Path(str(output_base) + ".parquet")
out_jsonl = Path(str(output_base) + ".jsonl")
out_errors = Path(str(output_base) + "_errors.parquet")
out_meta = Path(str(output_base) + "_meta.json")

# Parquet
try:
    df_out.to_parquet(out_parquet, index=False)
    print("Wrote:", out_parquet)
except Exception as e:
    print("Parquet write failed:", e)

# JSONL
with out_jsonl.open("w", encoding="utf-8") as f:
    for rec in df_out.to_dict(orient="records"):
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print("Wrote:", out_jsonl)

# Ошибки отдельным файлом
err_df = df_out[df_out["error"].notna()].copy()
if len(err_df):
    try:
        err_df.to_parquet(out_errors, index=False)
        print("Wrote:", out_errors)
    except Exception as e:
        print("Errors parquet write failed:", e)
else:
    print("Ошибок нет — отдельный файл не создан")

# Метаданные запуска
meta = {
    "run_config": asdict(run_config),
    "sample_size": int(len(df_sample)),
    "sample_ids": df_sample["id"].astype(str).tolist(),
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
with out_meta.open("w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print("Wrote:", out_meta)

Wrote: d:\project\multicriteria-dialog-audit-ml\data\transcripts_whisper_large_v3_1pct.parquet
Wrote: d:\project\multicriteria-dialog-audit-ml\data\transcripts_whisper_large_v3_1pct.jsonl
Ошибок нет — отдельный файл не создан
Wrote: d:\project\multicriteria-dialog-audit-ml\data\transcripts_whisper_large_v3_1pct_meta.json
